# B1 · 张量基础

**学完这一节你应该能**
- 看到一个 `shape`，说出每个数字代表什么
- 知道 `reshape` / `permute` / `unsqueeze` 各在干什么
- 知道广播什么时候会坑你
- 知道 `@` 和 `*` 的区别

**怎么用**
1. 从上往下跑，每格 `Shift + Enter`
2. 最后有 5 道练习，是**空着的**。改标了 `← 改这一行` 的地方，再跑那格
3. 5 个 `✓` 就算过

> 卡住了先看上面的讲解，每道题也带提示。

In [ ]:
import torch

print("torch :", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")


def check(name, ok, hint=""):
    """练习判定：ok 为 True 打 ✓，否则打 ✗ 并给提示"""
    if ok:
        print(f"✓ {name}")
    else:
        print(f"✗ {name}")
        if hint:
            print(f"   提示：{hint}")

## 1. 张量的三件套：`shape` / `dtype` / `device`

任何一个张量，先看这三样。90% 的报错都能从这三样里看出来。

In [ ]:
x = torch.randn(2, 3)
print(x)
print()
print("shape :", tuple(x.shape), "  ← 两个数字分别是什么？")
print("dtype :", x.dtype)
print("device:", x.device)

### 怎么读 shape

**从右往左读，最右边是最细的一层。**

| shape | 意思 |
|---|---|
| `(2, 3)` | 2 行 3 列的矩阵 |
| `(2, 3, 4)` | 2 个「3 行 4 列」的矩阵 |
| `(8, 16, 64, 32)` | 8 个 batch × 16 个 head × 64 个 token × 每 token 32 维 |

最后那个 `(8, 16, 64, 32)` 就是 transformer 里 attention 的典型形状，现在看不懂没关系，B3 之后会认识它。

In [ ]:
print("dtype 常见取值：")
for dt in [torch.float32, torch.float16, torch.bfloat16, torch.int64, torch.bool]:
    n = torch.tensor([], dtype=dt).element_size()
    print(f"  {str(dt):<20} 每个元素占 {n} 字节")

**`device`**：CPU 上是 `cpu`，GPU 上是 `cuda:0`。

两边**不能直接运算**，会报 `Expected all tensors to be on the same device`。解法是先 `.to(device)`。

代码里习惯性写成 `device = torch.device(...)`，换到 GPU 时一行都不用改。

## 2. 常用创建方式

In [ ]:
print("zeros(2, 3)      ="); print(torch.zeros(2, 3))
print("\nones(2, 3)     ="); print(torch.ones(2, 3))
print("\nfull((2,3), 7) ="); print(torch.full((2, 3), 7))
print("\narange(0, 10, 2)  =", torch.arange(0, 10, 2))
print("linspace(0, 1, 5) =", torch.linspace(0, 1, 5))
print("randint(0,10,(3,))=", torch.randint(0, 10, (3,)))
print("\nrandn(2, 3)      ="); print(torch.randn(2, 3))
print("  ↑ 标准正态：均值 0，方差 1。神经网络的初始化全用它。")

### `manual_seed` 是干什么的

固定随机数，保证两次跑出一样的结果。调试时非常有用 —— 否则你改了一处，结果变了，你分不清是改动导致的还是随机导致的。

In [ ]:
torch.manual_seed(0)
print("设种子后 :", torch.randn(3))

torch.manual_seed(0)
print("再设一次 :", torch.randn(3), "  ← 一样")

print("\n不设种子 :", torch.randn(3), "  ← 每次都不同")

## 3. 变形：`reshape` / `permute` / `unsqueeze`

这是读 transformer 代码时**最需要顺手的操作** —— 你会看到大量 `x.reshape(B, T, H, D)` 这类写法。

In [ ]:
a = torch.arange(12)
print("a =", a, " shape =", tuple(a.shape))

print("\na.reshape(3, 4) =")
print(a.reshape(3, 4))

print("\na.reshape(2, 2, 3).shape =", tuple(a.reshape(2, 2, 3).shape))

### `-1`：让 PyTorch 自己算

`reshape(3, -1)` 意思是「分成 3 份，每份多宽你自己算」。

In [ ]:
print("a.reshape(3, -1).shape =", tuple(a.reshape(3, -1).shape))
print(a.reshape(3, -1))

### `view` vs `reshape`

- `view` 只能在**内存连续**时用，否则报错
- `reshape` 会自动拷贝，永远能用

新手建议：**一律用 `reshape`**，别纠结。

In [ ]:
# 这个能跑
print(a.reshape(3, 4).shape)

# permute 之后内存就不连续了，view 会报错
img = torch.randn(3, 32, 32)
try:
    img.permute(1, 2, 0).view(-1)
except Exception as e:
    print("\nview 报错：", type(e).__name__)
    print("reshape 可以：", img.permute(1, 2, 0).reshape(-1).shape)

### `permute`：换维度顺序

PyTorch 的图像惯例是 `(通道, 高, 宽)`，而很多库要 `(高, 宽, 通道)`，就得 `permute`。

注意：`permute` **不移动数据**，只改「怎么读」。

In [ ]:
print("img               :", tuple(img.shape), " ← (C, H, W)")
print("img.permute(1,2,0):", tuple(img.permute(1, 2, 0).shape), " ← (H, W, C)")
print("img.transpose(0,1):", tuple(img.transpose(0, 1).shape), " ← 只换两个维度")

### `unsqueeze` / `squeeze`：加/去一个长度为 1 的维度

「凑 batch 维度」时天天用。模型要 `(batch, features)`，而你手上只有 `(features,)`，就 `unsqueeze(0)`。

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0])
print("v              :", tuple(v.shape))
print("v.unsqueeze(0) :", tuple(v.unsqueeze(0).shape), " ← 1 行 3 列")
print("v.unsqueeze(1) :", tuple(v.unsqueeze(1).shape), " ← 3 行 1 列")

t = torch.zeros(1, 5, 1)
print("\nsqueeze 去掉所有长度 1 的维度：", tuple(t.squeeze().shape))

## 4. 索引与切片

和 Python list 几乎一样，多了一个**布尔索引**。

In [ ]:
m = torch.arange(12).reshape(3, 4)
print(m)
print()
print("m[0]      =", m[0], "   ← 第 0 行")
print("m[:, 0]   =", m[:, 0], "   ← 第 0 列")
print("m[1:, 2:] ="); print(m[1:, 2:])
print("m[m > 5]  =", m[m > 5], " ← 布尔索引：取出所有大于 5 的")

idx = torch.tensor([0, 2])
print("m[idx]    ="); print(m[idx])

## 5. 广播（最常见的坑）

**规则一句话：从右往左逐维比较，要么相等，要么其中一个是 1，否则报错。**

广播让你少写循环，但也让你在写错 shape 时**不报错、静默算出错误结果**。这是它最危险的地方。

In [ ]:
x = torch.randn(3, 4)
bias = torch.randn(4)
print(f"x {tuple(x.shape)} + bias {tuple(bias.shape)} -> {tuple((x + bias).shape)}")
print("  bias 被自动复制成 3 行，逐行相加")

col = torch.randn(3, 1)
print(f"\nx {tuple(x.shape)} + col {tuple(col.shape)} -> {tuple((x + col).shape)}")
print("  col 里的 1 被自动复制成 4 列")

### 坑在这里

你以为在做「逐元素相加」，实际在造一个 3×4 的矩阵 —— 而且**不报错**。

In [ ]:
a = torch.randn(3, 1)
b = torch.randn(1, 4)
print(f"a {tuple(a.shape)} + b {tuple(b.shape)} -> {tuple((a + b).shape)}   ← 变成 3×4 了！")
print("\n想逐元素加，两个 shape 必须一模一样。")

In [ ]:
# 真正会报错的情况：从右往左，既不等也不是 1
try:
    torch.randn(3, 4) + torch.randn(3,)
except Exception as e:
    print("报错：", str(e).splitlines()[0])

## 6. `@` 和 `*`：两个完全不同的东西

这是初学者最容易混的一对。`*` 逐元素，`@` 矩阵乘。神经网络里**几乎全是 `@`**。

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

print("a ="); print(a)
print("\nb ="); print(b)
print("\na * b （逐元素） ="); print(a * b)
print("\na @ b （矩阵乘） ="); print(a @ b)

### 矩阵乘的 shape 规则

$$(m,\; k) \times (k,\; n) \rightarrow (m,\; n)$$

**中间那个 `k` 必须相等**，也就是「第一个的列数 = 第二个的行数」。

`nn.Linear(in=4, out=8)` 内部就是 `x @ W.T + b`，`W` 的形状是 `(8, 4)`。

In [ ]:
v = torch.randn(4)
w = torch.randn(8, 4)      # nn.Linear(4, 8) 的权重形状

print("v        :", tuple(v.shape))
print("w        :", tuple(w.shape), " ← (out, in)")
print("w.T      :", tuple(w.T.shape), " ← (in, out)")
print("v @ w.T  :", tuple((v @ w.T).shape), " ← 输出 8 维")

# 等价验证：用 F.linear 和手写对比
import torch.nn.functional as F
out1 = F.linear(v, w)
out2 = v @ w.T
print("\nF.linear 和手写差多少：", (out1 - out2).abs().max().item())

## 7. 练习

下面 5 道题，每道都是**故意错的**。改掉标了 `← 改这一行` 的那行，看到 `✓` 就过了。

In [ ]:
# 练习 1：t 的 shape 是 (2, 3, 4)，取出第 0 个 batch，结果应为 (3, 4)
t = torch.arange(24).reshape(2, 3, 4)
print("t.shape =", tuple(t.shape))

first = t                       # ← 改这一行

check("练习 1：shape 应为 (3, 4)", tuple(first.shape) == (3, 4),
      hint="沿第 0 维切一片：t[?]")

In [ ]:
# 练习 2：把 (2, 3, 4) 压成 (6, 4)
flat = t                        # ← 改这一行

check("练习 2：shape 应为 (6, 4)", tuple(flat.shape) == (6, 4),
      hint="t.reshape(6, 4)")

In [ ]:
# 练习 3：把 (3, 4) 转置成 (4, 3)
m = torch.zeros(3, 4)
t_m = m                         # ← 改这一行

check("练习 3：shape 应为 (4, 3)", tuple(t_m.shape) == (4, 3),
      hint="矩阵转置用 .T 或 .transpose(0, 1)")

In [ ]:
# 练习 4：x 是 (3, 4)，vec 是 (4,)，让每一行都加上 vec
# 结果第 0 行应该变成 [2, 3, 4, 5]
x = torch.ones(3, 4)
vec = torch.tensor([1.0, 2.0, 3.0, 4.0])

out = x                         # ← 改这一行

check("练习 4：shape (3,4) 且第 0 行为 [2,3,4,5]",
      tuple(out.shape) == (3, 4) and torch.allclose(out[0], torch.tensor([2., 3., 4., 5.])),
      hint="x + vec 就够了。想想为什么不用管行数？")

In [ ]:
# 练习 5：区分 * 和 @
# A 是 (2,3)，B 是 (3,)
#   mul 应为逐元素乘（广播后 (2,3)）
#   mat 应为矩阵乘（(2,)）
A = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
B = torch.tensor([1., 10., 100.])

mul = A                         # ← 改这一行
mat = A                         # ← 改这一行

check("练习 5：区分 * 和 @",
      tuple(mul.shape) == (2, 3)
      and tuple(mat.shape) == (2,)
      and torch.allclose(mul[0], torch.tensor([1., 20., 300.]))
      and torch.allclose(mat, torch.tensor([321., 654.])),
      hint="mul 用 *，mat 用 @。注意 mat 是 (2,) 不是 (2,1)")

## 收尾

全 `✓` 之后，回答这三个，答出来才算真懂：

1. `(8, 16, 64, 32)` 这四个数字分别代表什么？
2. 为什么说 `reshape` 比 `view` 安全？
3. 什么情况下 `a * b` 和 `a @ b` 的形状一样、但结果完全不同？

下一节：**B2 · autograd**。